# 02 — Efficiency estimators and the auto-router

anchor-op ships three efficiency estimators + an `"auto"` router that picks per data format. This
notebook shows what each estimates, when each is appropriate, and how the `min_control_detection_rate`
filter protects against information-limited targets.

In [ ]:
import numpy as np
import matplotlib.pyplot as plt
import anchorop as ao
rng = np.random.default_rng(0)

## The three estimators

- **`estimate_knockdown_efficiency`** (`mean_ratio`): `κ = 1 − mean_pert / mean_ctrl`. The sample-moment
  MLE under Poisson observation. Also unbiased under independent zero-inflation (dropout cancels in
  the ratio). Its finite-sample failure is at very low baseline expression where the ratio-of-means
  distribution becomes bimodal and spikes to 0 or 1.
- **`estimate_knockdown_efficiency_poisson_mle`** (`poisson_mle`): `κ = 1 − λ̂_pert / λ̂_ctrl` with
  `λ̂ = −log(1 − detection_rate)`. Equivalent to `mean_ratio` under pure Poisson; biased downward
  under independent zero-inflation.
- **`estimate_knockdown_efficiency_detection_rate`** (`detection_rate`): the raw shift
  `Pr[X_ctrl > 0] − Pr[X_pert > 0]`. On count data this is NOT an unbiased estimator of κ. On
  pre-scaled residual data (where controls have mean ≈ 0 by z-scoring construction) it becomes a
  valid signed distributional-shift statistic ≈ `0.5 − Φ(Δ/σ)`.

## Demo 1: count data, all three estimators agree at moderate expression

In [ ]:
def draw_count_data(lam_ctrl, kappa, n_ctrl=300, n_pert=80, rng=rng):
    ctrl = rng.poisson(lam_ctrl, size=n_ctrl).astype(float)
    pert = rng.poisson(max(1e-9, (1 - kappa) * lam_ctrl), size=n_pert).astype(float)
    X = np.zeros((n_ctrl + n_pert, 1))
    X[:n_ctrl, 0] = ctrl; X[n_ctrl:, 0] = pert
    cm = np.zeros(n_ctrl + n_pert, dtype=bool); cm[:n_ctrl] = True
    return X, cm, ~cm

X, cm, pm = draw_count_data(lam_ctrl=3.0, kappa=0.5)
print(f"κ_true = 0.5, baseline λ = 3.0 (moderate)")
print(f"  mean_ratio:      {ao.estimate_knockdown_efficiency(X, target_index=0, perturbed_mask=pm, control_mask=cm):.3f}")
print(f"  poisson_mle:     {ao.estimate_knockdown_efficiency_poisson_mle(X, target_index=0, perturbed_mask=pm, control_mask=cm):.3f}")
print(f"  detection_rate:  {ao.estimate_knockdown_efficiency_detection_rate(X, target_index=0, perturbed_mask=pm, control_mask=cm):.3f}")

## Demo 2: low-baseline dropout regime, estimators disagree

In [ ]:
X, cm, pm = draw_count_data(lam_ctrl=0.05, kappa=0.5)   # near-zero baseline
print(f"κ_true = 0.5, baseline λ = 0.05 (near dropout floor)")
print(f"  mean_ratio:      {ao.estimate_knockdown_efficiency(X, target_index=0, perturbed_mask=pm, control_mask=cm):.3f}  ← bimodal/spike")
print(f"  poisson_mle:     {ao.estimate_knockdown_efficiency_poisson_mle(X, target_index=0, perturbed_mask=pm, control_mask=cm):.3f}")
print(f"  detection_rate:  {ao.estimate_knockdown_efficiency_detection_rate(X, target_index=0, perturbed_mask=pm, control_mask=cm):.3f}  ← bounded shift, not κ")

The `mean_ratio` estimate is dominated by discretization noise in the near-dropout regime. This
is what the `min_control_detection_rate` filter protects against — see the next demo.

## Demo 3: pre-scaled (z-scored) residual data — detection_rate is analytically valid, mean_ratio blows up

On z-scored data, `Pr[X > 0]` in controls is ≈ 0.5 by construction, and a perturbation that shifts
the distribution downward reduces `Pr[X > 0]` in perturbed cells. So detection_rate becomes a signed
distributional-shift statistic ≈ `0.5 − Φ(Δ/σ_ctrl)`. Meanwhile mean_ratio is undefined because
the control mean is ~0.

In [ ]:
def draw_zscored_data(shift, n_ctrl=300, n_pert=80, rng=rng):
    ctrl = rng.normal(0.0, 1.0, size=n_ctrl)
    pert = rng.normal(shift, 1.0, size=n_pert)
    X = np.zeros((n_ctrl + n_pert, 1))
    X[:n_ctrl, 0] = ctrl; X[n_ctrl:, 0] = pert
    cm = np.zeros(n_ctrl + n_pert, dtype=bool); cm[:n_ctrl] = True
    return X, cm, ~cm

X, cm, pm = draw_zscored_data(shift=-1.0)  # knockdown = downward shift of 1σ
print("z-scored data, Δ = -1σ:")
print(f"  detection_rate:  {ao.estimate_knockdown_efficiency_detection_rate(X, target_index=0, perturbed_mask=pm, control_mask=cm):.3f}")
print(f"  mean_ratio:      {ao.estimate_knockdown_efficiency(X, target_index=0, perturbed_mask=pm, control_mask=cm):.3f}  ← unstable/undefined")

## The auto router

`build_guide_responses(..., efficiency_estimator="auto")` inspects the expression matrix and picks
`mean_ratio` on count-like data, `detection_rate` on pre-scaled residual data (matrix has ≥ 2%
negative entries). This is the recommended default. You can pin an explicit estimator by passing
its name.

## The min_control_detection_rate filter

Drops targets whose control detection rate is below the threshold (default 0.05). On count data
this protects against the finite-sample pathology above. On pre-scaled data the filter is inert
(all genes have ~50% positive-value fraction).

## Which estimator on which data?

- Raw or normalized-count Perturb-seq → `auto` → `mean_ratio` + filter.
- Pre-scaled residual h5ads (Replogle 2022 essential-gene style) → `auto` → `detection_rate` as a
  signed shift statistic. The manuscript §2.3 explains what this quantity actually estimates on
  that data class.

## Next: 03 for the full measurement pipeline on an AnnData object.